# Tidy Data: melt, pivot, split, unite

*From my BYU-Idaho DS 250 "Week 10" tidy-data lesson. This is a technique-demo notebook I
typed following the course material (which ports R's `tidyr` teaching examples to pandas via
[byuidatascience/python4ds](https://byuidatascience.github.io/python4ds/tidy-data.html)); the
lesson structure and the `table1`–`table5` example datasets are the course's, the pandas code
is mine.*

The whole notebook runs on small **public** teaching datasets pulled straight from the
byuidatascience GitHub (the classic WHO-TB `table1`–`table5`, plus `mpg`/`presidential`), so
it executes top-to-bottom with no local files. The point is to see the four reshape verbs on
data small enough to eyeball every row:

- **`melt`** — wide → long (spread-out year columns collapse into one `year` column)
- **`pivot_table`** — long → wide (a `type` column of key/value rows fans back out to columns)
- **split** — one column into several (`.str.split(expand=True)`)
- **unite** — several columns into one (`.agg('-'.join, axis=1)`)

In [1]:
import pandas as pd
import altair as alt
import numpy as np


In [2]:
base_url = "https://github.com/byuidatascience/data4python4ds/raw/master/data-raw/"
table1 = pd.read_csv("{}table1/table1.csv".format(base_url))
table2 = pd.read_csv("{}table2/table2.csv".format(base_url))
table3 = pd.read_csv("{}table3/table3.csv".format(base_url))
table4a = pd.read_csv("{}table4a/table4a.csv".format(base_url))
table4b = pd.read_csv("{}table4b/table4b.csv".format(base_url))
table5 = pd.read_csv("{}table5/table5.csv".format(base_url), dtype = 'object')


In [3]:
table1

,country,year,cases,population
0,Afghanistan,1999,745,19987071
1,Afghanistan,2000,2666,20595360
2,Brazil,1999,37737,172006362
3,Brazil,2000,80488,174504898
4,China,1999,212258,1272915272
5,China,2000,213766,1280428583


In [4]:
table2

,country,year,type,count
0,Afghanistan,1999,cases,745
1,Afghanistan,1999,population,19987071
2,Afghanistan,2000,cases,2666
3,Afghanistan,2000,population,20595360
4,Brazil,1999,cases,37737
5,Brazil,1999,population,172006362
6,Brazil,2000,cases,80488
7,Brazil,2000,population,174504898
8,China,1999,cases,212258
9,China,1999,population,1272915272


In [5]:
table3

,country,year,rate
0,Afghanistan,1999,745/19987071
1,Afghanistan,2000,2666/20595360
2,Brazil,1999,37737/172006362
3,Brazil,2000,80488/174504898
4,China,1999,212258/1272915272
5,China,2000,213766/1280428583


In [6]:
table4a

,country,1999,2000
0,Afghanistan,745,2666
1,Brazil,37737,80488
2,China,212258,213766


In [7]:
table4b

,country,1999,2000
0,Afghanistan,19987071,20595360
1,Brazil,172006362,174504898
2,China,1272915272,1280428583


In [8]:
table5

,country,century,year,rate
0,Afghanistan,19,99,745/19987071
1,Afghanistan,20,00,2666/20595360
2,Brazil,19,99,37737/172006362
3,Brazil,20,00,80488/174504898
4,China,19,99,212258/1272915272
5,China,20,00,213766/1280428583


In [9]:
table1.assign(
    rate = lambda row: row.cases / row.population * 100
)

,country,year,cases,population,rate
0,Afghanistan,1999,745,19987071,0.003727
1,Afghanistan,2000,2666,20595360,0.012945
2,Brazil,1999,37737,172006362,0.021939
3,Brazil,2000,80488,174504898,0.046124
4,China,1999,212258,1272915272,0.016675
5,China,2000,213766,1280428583,0.016695


In [10]:

df = (table1.
  groupby('year').
  agg(n = ('cases', 'sum'))).reset_index()
df


,year,n
0,1999,250740
1,2000,296920


In [11]:
(alt.Chart(df)
 .mark_bar()
 .encode(
     alt.X('year'),
     alt.Y('n')
 ))

alt.Chart(...)

In [12]:

(table1.
  groupby('year').
  agg(n = ('cases', 'sum')).
  reset_index())



,year,n
0,1999,250740
1,2000,296920


In [13]:
base_chart = (alt.Chart(table1).
  encode(alt.X('year'), alt.Y('cases'), detail = 'country'))

chart = base_chart.mark_line() + base_chart.encode(color = 'country').mark_circle()
chart
# chart.save("screenshots/altair_table1.png")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/altair/utils/core.py:384: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.LayerChart(...)

## function melt()

A common problem is a dataset where some of the column names are not names of variables, but values of a variable. Take table4a: the column names 1999 and 2000 represent values of the year variable, the values in the 1999 and 2000 columns represent values of the cases variable, and each row represents two observations, not one.

In [14]:
table4a

,country,1999,2000
0,Afghanistan,745,2666
1,Brazil,37737,80488
2,China,212258,213766


In [15]:
table4a.melt(['country'], var_name = "year", value_name = "cases")

,country,year,cases
0,Afghanistan,1999,745
1,Brazil,1999,37737
2,China,1999,212258
3,Afghanistan,2000,2666
4,Brazil,2000,80488
5,China,2000,213766


In [16]:
t4a = table4a.copy()
t4a = t4a.assign(population=[100, 200, 300])
t4a

,country,1999,2000,population
0,Afghanistan,745,2666,100
1,Brazil,37737,80488,200
2,China,212258,213766,300


In [17]:
t4a.melt(['country', 'population'], var_name = "year", value_name = "cases")

,country,population,year,cases
0,Afghanistan,100,1999,745
1,Brazil,200,1999,37737
2,China,300,1999,212258
3,Afghanistan,100,2000,2666
4,Brazil,200,2000,80488
5,China,300,2000,213766


In [18]:
# Table 4b
table4b

,country,1999,2000
0,Afghanistan,19987071,20595360
1,Brazil,172006362,174504898
2,China,1272915272,1280428583


In [19]:
table4b.melt(['country'], var_name = "year", value_name = "population")

,country,year,population
0,Afghanistan,1999,19987071
1,Brazil,1999,172006362
2,China,1999,1272915272
3,Afghanistan,2000,20595360
4,Brazil,2000,174504898
5,China,2000,1280428583


## function pivot()

pivot() is the opposite of melt(). You use it when an observation is scattered across multiple rows. For example, take table2: an observation is a country in a year, but each observation is spread across two rows.

In [20]:
table2

,country,year,type,count
0,Afghanistan,1999,cases,745
1,Afghanistan,1999,population,19987071
2,Afghanistan,2000,cases,2666
3,Afghanistan,2000,population,20595360
4,Brazil,1999,cases,37737
5,Brazil,1999,population,172006362
6,Brazil,2000,cases,80488
7,Brazil,2000,population,174504898
8,China,1999,cases,212258
9,China,1999,population,1272915272


In [21]:
table2.pivot_table(
    index = ['country', 'year'], 
    columns = 'type', 
    values = 'count').reset_index()

type,country,year,cases,population
0,Afghanistan,1999,745.0,1.998707e+07
1,Afghanistan,2000,2666.0,2.059536e+07
2,Brazil,1999,37737.0,1.720064e+08
3,Brazil,2000,80488.0,1.745049e+08
4,China,1999,212258.0,1.272915e+09
5,China,2000,213766.0,1.280429e+09


## function split()

str.split() pulls apart one column into multiple columns, by splitting wherever a separator character appears. Take table3:

In [22]:
table3

,country,year,rate
0,Afghanistan,1999,745/19987071
1,Afghanistan,2000,2666/20595360
2,Brazil,1999,37737/172006362
3,Brazil,2000,80488/174504898
4,China,1999,212258/1272915272
5,China,2000,213766/1280428583


In [23]:
print('hi there and are happy'.split())

['hi', 'there', 'and', 'are', 'happy']


In [24]:
new_columns = (table3.rate.str.split("/", expand = True))
new_columns 


,0,1
0,745,19987071
1,2666,20595360
2,37737,172006362
3,80488,174504898
4,212258,1272915272
5,213766,1280428583


In [25]:
new_columns = (table3.
  rate.str.split("/", expand = True).
  rename(columns = {0: "cases", 1: "population"})
  )
new_columns 


,cases,population
0,745,19987071
1,2666,20595360
2,37737,172006362
3,80488,174504898
4,212258,1272915272
5,213766,1280428583


In [26]:
new_table = pd.concat([table3.drop(columns = 'rate'), new_columns], axis = 1)
new_table

,country,year,cases,population
0,Afghanistan,1999,745,19987071
1,Afghanistan,2000,2666,20595360
2,Brazil,1999,37737,172006362
3,Brazil,2000,80488,174504898
4,China,1999,212258,1272915272
5,China,2000,213766,1280428583


In [27]:
t3 = table3.copy()
t3[['cases', 'population']] = t3['rate'].str.split('/', expand=True)
t3 = t3.drop(['rate'], axis=1)
t3

,country,year,cases,population
0,Afghanistan,1999,745,19987071
1,Afghanistan,2000,2666,20595360
2,Brazil,1999,37737,172006362
3,Brazil,2000,80488,174504898
4,China,1999,212258,1272915272
5,China,2000,213766,1280428583


## function apply()


In [28]:
table3

,country,year,rate
0,Afghanistan,1999,745/19987071
1,Afghanistan,2000,2666/20595360
2,Brazil,1999,37737/172006362
3,Brazil,2000,80488/174504898
4,China,1999,212258/1272915272
5,China,2000,213766/1280428583


In [29]:
t3 = table3.copy()
t3['cases'] = t3['rate'].apply(lambda x: x.split('/')[0])
t3['population'] = t3['rate'].apply(lambda x: x.split('/')[1])
t3 = t3.drop(['rate'], axis=1)
t3

,country,year,cases,population
0,Afghanistan,1999,745,19987071
1,Afghanistan,2000,2666,20595360
2,Brazil,1999,37737,172006362
3,Brazil,2000,80488,174504898
4,China,1999,212258,1272915272
5,China,2000,213766,1280428583


In [30]:
t3
t3['cases'].apply(lambda case: int(case) / 2)



0       372.5
1      1333.0
2     18868.5
3     40244.0
4    106129.0
5    106883.0
Name: cases, dtype: float64

In [31]:
def process_case(case):
    number = int(case)
    if (number < 1000):
        return 0
    else:
        return 1

t3
t3['over1000'] = t3['cases'].apply(process_case)
t3


,country,year,cases,population,over1000
0,Afghanistan,1999,745,19987071,0
1,Afghanistan,2000,2666,20595360,1
2,Brazil,1999,37737,172006362,1
3,Brazil,2000,80488,174504898,1
4,China,1999,212258,1272915272,1
5,China,2000,213766,1280428583,1


## function Unite()

For two string series the inverse of str.split() can be done with +: it combines multiple columns into a single column. You’ll need it much less frequently than str.split(), but it’s still a useful tool to have in your back pocket.

In [32]:
table5

,country,century,year,rate
0,Afghanistan,19,99,745/19987071
1,Afghanistan,20,00,2666/20595360
2,Brazil,19,99,37737/172006362
3,Brazil,20,00,80488/174504898
4,China,19,99,212258/1272915272
5,China,20,00,213766/1280428583


In [33]:
table5.assign(new = table5['century'] + table5['year'])


,country,century,year,rate,new
0,Afghanistan,19,99,745/19987071,1999
1,Afghanistan,20,00,2666/20595360,2000
2,Brazil,19,99,37737/172006362,1999
3,Brazil,20,00,80488/174504898,2000
4,China,19,99,212258/1272915272,1999
5,China,20,00,213766/1280428583,2000


In [34]:
table5.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   country  6 non-null      object
 1   century  6 non-null      object
 2   year     6 non-null      object
 3   rate     6 non-null      object
dtypes: object(4)
memory usage: 320.0+ bytes


In [35]:
table5.assign(new = table5[['century', 'year']].agg("_".join, axis = 1))

,country,century,year,rate,new
0,Afghanistan,19,99,745/19987071,19_99
1,Afghanistan,20,00,2666/20595360,20_00
2,Brazil,19,99,37737/172006362,19_99
3,Brazil,20,00,80488/174504898,20_00
4,China,19,99,212258/1272915272,19_99
5,China,20,00,213766/1280428583,20_00


In [36]:
# split()
words = 'hi there i am 10 years old'
parts = words.split()

In [37]:
' - '.join(parts)

'hi - there - i - am - 10 - years - old'

In [38]:
# Python's join function()

letters = ['h', 'i', ' ', 't', 'h', 'e', 'r', 'e']
letters

['h', 'i', ' ', 't', 'h', 'e', 'r', 'e']

---
# Altair

In [39]:
import altair as alt
import pandas as pd
import numpy as np
url_1 = "https://github.com/byuidatascience/data4python4ds/raw/master/data-raw/mpg/mpg.csv"
url_2 = "https://github.com/byuidatascience/data4python4ds/raw/master/data-raw/presidential/presidential.csv"
url_3 = "https://github.com/byuidatascience/data4python4ds/raw/master/data-raw/diamonds/diamonds.csv"

mpg = pd.read_csv(url_1)
presidential = pd.read_csv(url_2)
diamonds = pd.read_csv(url_3)


In [40]:
presidential


,name,start,end,party
0,Eisenhower,1953-01-20,1961-01-20,Republican
1,Kennedy,1961-01-20,1963-11-22,Democratic
2,Johnson,1963-11-22,1969-01-20,Democratic
3,Nixon,1969-01-20,1974-08-09,Republican
4,Ford,1974-08-09,1977-01-20,Republican
5,Carter,1977-01-20,1981-01-20,Democratic
6,Reagan,1981-01-20,1989-01-20,Republican
7,Bush,1989-01-20,1993-01-20,Republican
8,Clinton,1993-01-20,2001-01-20,Democratic
9,Bush,2001-01-20,2009-01-20,Republican


In [41]:
presidential.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   name    11 non-null     object
 1   start   11 non-null     object
 2   end     11 non-null     object
 3   party   11 non-null     object
dtypes: object(4)
memory usage: 480.0+ bytes


In [42]:

presidential = presidential.assign(
    start = pd.to_datetime(presidential.start),
    end = pd.to_datetime(presidential.end),
    id = 33 + presidential.index
)
presidential

,name,start,end,party,id
0,Eisenhower,1953-01-20,1961-01-20,Republican,33
1,Kennedy,1961-01-20,1963-11-22,Democratic,34
2,Johnson,1963-11-22,1969-01-20,Democratic,35
3,Nixon,1969-01-20,1974-08-09,Republican,36
4,Ford,1974-08-09,1977-01-20,Republican,37
5,Carter,1977-01-20,1981-01-20,Democratic,38
6,Reagan,1981-01-20,1989-01-20,Republican,39
7,Bush,1989-01-20,1993-01-20,Republican,40
8,Clinton,1993-01-20,2001-01-20,Democratic,41
9,Bush,2001-01-20,2009-01-20,Republican,42


In [43]:
presidential.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   name    11 non-null     object        
 1   start   11 non-null     datetime64[ns]
 2   end     11 non-null     datetime64[ns]
 3   party   11 non-null     object        
 4   id      11 non-null     int64         
dtypes: datetime64[ns](2), int64(1), object(2)
memory usage: 568.0+ bytes


In [44]:
chart = (alt.Chart(mpg, 
            title = "Fuel efficiency generally decreases with engine size")
          .encode(alt.X('displ'), alt.Y('hwy'), color = 'class')
          .mark_circle())
chart


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/altair/utils/core.py:384: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/altair/utils/core.py:384: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/altair/utils/core.py:384: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_n

alt.Chart(...)

In [45]:
chart = (alt.Chart(mpg)
    .encode(alt.X('displ'), alt.Y('hwy'),color = 'class')
    .mark_circle()
    .properties(
      title = {
        "text":  "Fuel efficiency generally decreases with engine size",
        "subtitle": "Two seaters (sports cars) are an exception because of their light weight"
        })
    .configure_title(
      fontSize = 15, 
      anchor = "start", 
      subtitleFontSize = 11))

chart

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/altair/utils/core.py:384: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/altair/utils/core.py:384: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/altair/utils/core.py:384: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_n

alt.Chart(...)

In [46]:
chart = (alt.Chart(mpg)
    .encode(
      alt.X('displ', title = "Engine displacement (L)"), 
      alt.Y('hwy', title = "Highway fuel economy (mpg)"), 
      color = alt.Color('class', title = "Car type")
      )
    .mark_circle())

chart

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/altair/utils/core.py:384: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/altair/utils/core.py:384: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/altair/utils/core.py:384: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_n

alt.Chart(...)

In [47]:
best_in_class = (mpg
    .assign(
      hwy_min = lambda x: x.groupby('class').hwy.transform('max'))
    .query('(hwy_min == hwy)')
    .drop_duplicates('class', keep = 'first'))

base = (alt.Chart(mpg)
  .encode(
    alt.X('displ'), 
    alt.Y('hwy'),
    alt.Color('class')
    )
  .mark_circle())

text = (alt.Chart(best_in_class)
    .encode(
      alt.X('displ'), 
      alt.Y('hwy'),
      text = 'model'
      )
    .mark_text())

chart = base + text

chart

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/altair/utils/core.py:384: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/altair/utils/core.py:384: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/altair/utils/core.py:384: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_n

alt.LayerChart(...)

In [48]:
chart = (alt.Chart(mpg)
  .encode(
    alt.X('displ'), 
    alt.Y('hwy', axis = alt.Axis(
            values = np.arange(15, 40, step = 5).tolist()),
            scale = alt.Scale(zero = False )), 
    alt.Color('class'))
  .mark_circle())
    
chart

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/altair/utils/core.py:384: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/altair/utils/core.py:384: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/altair/utils/core.py:384: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_n

alt.Chart(...)

In [49]:
chart = (alt.Chart(mpg)
    .encode(
      alt.X('displ'), 
      alt.Y('hwy'), 
      color = alt.Color('drv', scale = alt.Scale(scheme = 'set1')),
      shape = alt.Shape('drv:N')
      )
    .mark_point(filled = True))

chart

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/altair/utils/core.py:384: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/altair/utils/core.py:384: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/altair/utils/core.py:384: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_n

alt.Chart(...)

---
# Strings

https://byuidatascience.github.io/python4ds/strings.html

In [50]:
first = ["Murry", "Bob", "April", "Mary"]
last = ["Brown", "Smith 2", "Johnson", "Brook"]
grade = [90, 40, 80, 98]
  
# dictionary of lists 
dict = {'firstname': first, 'surname': last, 'score': grade} 
    
df = pd.DataFrame(dict)
df

,firstname,surname,score
0,Murry,Brown,90
1,Bob,Smith 2,40
2,April,Johnson,80
3,Mary,Brook,98


In [51]:
df['surname'].str.capitalize()

0      Brown
1    Smith 2
2    Johnson
3      Brook
Name: surname, dtype: object

In [52]:
df['surname'].str.isalpha()

0     True
1    False
2     True
3     True
Name: surname, dtype: bool

In [53]:
df[['firstname', 'surname']].agg(len, axis=1)

0    2
1    2
2    2
3    2
dtype: int64

In [54]:
df[['firstname', 'surname']].agg('-'.join, axis=1)

0      Murry-Brown
1      Bob-Smith 2
2    April-Johnson
3       Mary-Brook
dtype: object

In [55]:
df['surname'].str.replace('Brook', 'Brooke')

0      Brown
1    Smith 2
2    Johnson
3     Brooke
Name: surname, dtype: object

In [56]:
df2 = df.query('surname.str.contains("B")')
df2

,firstname,surname,score
0,Murry,Brown,90
3,Mary,Brook,98


In [57]:
df.assign(
    first_vowels = df.firstname.str.count('[aeiou]'),
    surname_vowels = df.surname.str.count('[aeiou]'),
    surname_cons = df.surname.str.count('[^aeiou]'),
    surname_len = df.surname.str.len(),
)

,firstname,surname,score,first_vowels,surname_vowels,surname_cons,surname_len
0,Murry,Brown,90,1,1,4,5
1,Bob,Smith 2,40,1,1,6,7
2,April,Johnson,80,1,2,5,7
3,Mary,Brook,98,1,2,3,5


## Lead-in to the Star Wars case study

The course lesson ended by loading `StarWars.csv` — the FiveThirtyEight survey — as the jump
into the Project 5 cleaning task. That full case study (cleaning the Qualtrics column names,
ordinal-encoding the survey answers, one-hot encoding, building an ML target) lives on its own
page: **[Star Wars survey case study](../case-study-star-wars-survey.md)**. The raw CSV is not
committed here; the case-study page links the public FiveThirtyEight source.